####  Ticketing Tool 

### 2. Linear Routing

In [1]:
from IPython.display import display, HTML
linear_html = """
<div>
<h2>Concept 1 - Linear Workflow</h2>

<p>
A Jira ticket follows a fixed sequence:
Classification → Summary → Assignment.
</p>

<pre>
Ticket
 ↓
Classification
 ↓
Summary
 ↓
Assignment
</pre>

<p>
Every node executes once.
</p>

</div>
"""
display(HTML(linear_html))

In [2]:
from langgraph.graph import StateGraph
from typing import TypedDict, Dict, Any

class TicketState(TypedDict):
    issue_key:str
    summary:str
    issue_type:str
    category:str
    assignment_group:str

def classify_ticket(state):
    state["category"] = state["issue_type"]
    return state

def summarize_ticket(state):
    state["summary"] = state["summary"][:75]
    return state

def assign_team(state):
    state["assignment_group"] = "AMS Team"
    return state

builder = StateGraph(TicketState)

builder.add_node("classify_ticket", classify_ticket)
builder.add_node("summarize_ticket", summarize_ticket)
builder.add_node("assign_team", assign_team)

builder.set_entry_point("classify_ticket")

builder.add_edge("classify_ticket","summarize_ticket")
builder.add_edge("summarize_ticket","assign_team")

builder.set_finish_point("assign_team")

graph = builder.compile()
from IPython.display import Image, Markdown, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))

except Exception as e:

    print(f"Graph Visualization Failed: {e}")

    mermaid_code = graph.get_graph().draw_mermaid()

    display(
        Markdown(
            f"```mermaid\n{mermaid_code}\n```"
        )
    )

Graph Visualization Failed: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_ticket(classify_ticket)
	summarize_ticket(summarize_ticket)
	assign_team(assign_team)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_ticket;
	classify_ticket --> summarize_ticket;
	summarize_ticket --> assign_team;
	assign_team --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = graph.invoke({
    "issue_key":"SEIT-124577",
    "summary":"P1 DM not available",
    "issue_type":"Incident",
    "category":"",
    "assignment_group":""
})
print(result)
import pandas as pd
pd.DataFrame([result])

{'issue_key': 'SEIT-124577', 'summary': 'P1 DM not available', 'issue_type': 'Incident', 'category': 'Incident', 'assignment_group': 'AMS Team'}


,issue_key,summary,issue_type,category,assignment_group
0,SEIT-124577,P1 DM not available,Incident,Incident,AMS Team


### 2. Conditional Routing

In [9]:
from IPython.display import display, HTML
routing_html = """
<div>
<h2>Concept 2 - Conditional Routing</h2>

<pre>
Ticket
 ↓
Issue Type?
 /      \\
Incident Service Request
 ↓          ↓
AMS      Access Team
</pre>

</div>
"""
display(HTML(routing_html))

In [10]:
from langgraph.graph import StateGraph
from typing import TypedDict, Dict, Any
from IPython.display import Image, display

class TicketState(TypedDict):
    issue_type:str
    team:str
    
def router(state):
    return state

def ams_team(state):
    state["team"]="AMS Team"
    return state

def access_team(state):
    state["team"]="Access Team"
    return state

def route_ticket(state):

    if state["issue_type"] == "Incident":
        return "ams_team"

    return "access_team"

builder = StateGraph(TicketState)

builder.add_node("router", router)
builder.add_node("ams_team", ams_team)
builder.add_node("access_team", access_team)

builder.set_entry_point("router")

builder.add_conditional_edges(
    "router",
    route_ticket,
    {
        "ams_team":"ams_team",
        "access_team":"access_team"
    }
)

builder.set_finish_point("ams_team")
builder.set_finish_point("access_team")
graph = builder.compile()

from IPython.display import Image, Markdown, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))

except Exception as e:

    print(f"Graph Visualization Failed: {e}")

    mermaid_code = graph.get_graph().draw_mermaid()

    display(
        Markdown(
            f"```mermaid\n{mermaid_code}\n```"
        )
    )


Graph Visualization Failed: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	ams_team(ams_team)
	access_team(access_team)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> access_team;
	router -.-> ams_team;
	access_team --> __end__;
	ams_team --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [11]:
result1 = graph.invoke({
    "issue_type":"Incident",
    "team":""
})

In [12]:
result2 = graph.invoke({
    "issue_type":"Service Request",
    "team":""
})


In [13]:
pd.DataFrame([result1,result2])

,issue_type,team
0,Incident,AMS Team
1,Service Request,Access Team


#### Concept 3: Multiple Routers

In [18]:
from IPython.display import display, HTML
multiple_router_html = """
<div style="font-family:Segoe UI,Arial,sans-serif;line-height:1.6;">

    <h2 style="color:#0078d4;">
        Concept 3 - Multiple Routers
    </h2>

    <p>
        This example demonstrates how LangGraph can perform multiple
        routing decisions within the same workflow.
    </p>

    <h3>Business Scenario</h3>

    <p>
        A Jira ticket is first evaluated based on Issue Type.
        If it is an Incident, a second decision point evaluates
        Priority and assigns it to the appropriate support team.
    </p>

    <pre style="background:#f4f4f4;padding:10px;">
Ticket Raised
      ↓
Issue Type?
      ↓
 Incident
      ↓
 Priority?
   /       \\
 High      Low
   ↓         ↓
L2 Team   L1 Team
    </pre>

    <p>
        Multiple routers help enterprise systems apply
        multiple business rules before assigning work.
    </p>

</div>
"""
display(HTML(multiple_router_html))

In [20]:
from typing import TypedDict

class TicketState(TypedDict):
    issue_key: str
    issue_type: str
    priority: str
    assignment_group: str
def issue_router(state):
    return state


def priority_router(state):
    return state


def assign_l1(state):

    state["assignment_group"] = "L1 Support Team"

    return state


def assign_l2(state):

    state["assignment_group"] = "L2 Support Team"

    return state
def route_issue_type(state):

    if state["issue_type"] == "Incident":
        return "priority_router"

    return "assign_l1"


def route_priority(state):

    if state["priority"] == "High":
        return "assign_l2"

    return "assign_l1"
from langgraph.graph import StateGraph

builder = StateGraph(TicketState)

builder.add_node(
    "issue_router",
    issue_router
)

builder.add_node(
    "priority_router",
    priority_router
)

builder.add_node(
    "assign_l1",
    assign_l1
)

builder.add_node(
    "assign_l2",
    assign_l2
)

builder.set_entry_point(
    "issue_router"
)

builder.add_conditional_edges(
    "issue_router",
    route_issue_type,
    {
        "priority_router": "priority_router",
        "assign_l1": "assign_l1"
    }
)

builder.add_conditional_edges(
    "priority_router",
    route_priority,
    {
        "assign_l1": "assign_l1",
        "assign_l2": "assign_l2"
    }
)

builder.set_finish_point("assign_l1")
builder.set_finish_point("assign_l2")

graph = builder.compile()
from IPython.display import Image, Markdown, display

try:

    display(
        Image(
            graph.get_graph().draw_mermaid_png()
        )
    )

except Exception as e:

    print(f"Graph Visualization Failed: {e}")

    mermaid_code = graph.get_graph().draw_mermaid()

    display(
        Markdown(
            f"```mermaid\n{mermaid_code}\n```"
        )
    )


Graph Visualization Failed: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	issue_router(issue_router)
	priority_router(priority_router)
	assign_l1(assign_l1)
	assign_l2(assign_l2)
	__end__([<p>__end__</p>]):::last
	__start__ --> issue_router;
	issue_router -.-> assign_l1;
	issue_router -.-> priority_router;
	priority_router -.-> assign_l1;
	priority_router -.-> assign_l2;
	assign_l1 --> __end__;
	assign_l2 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [21]:
result1 = graph.invoke({

    "issue_key": "SEIT-124577",
    "issue_type": "Incident",
    "priority": "High",
    "assignment_group": ""

})

print(result1)

{'issue_key': 'SEIT-124577', 'issue_type': 'Incident', 'priority': 'High', 'assignment_group': 'L2 Support Team'}


In [22]:
result2 = graph.invoke({

    "issue_key": "SEIT-124711",
    "issue_type": "Incident",
    "priority": "Low",
    "assignment_group": ""

})

print(result2)

{'issue_key': 'SEIT-124711', 'issue_type': 'Incident', 'priority': 'Low', 'assignment_group': 'L1 Support Team'}


In [23]:
result3 = graph.invoke({

    "issue_key": "SEIT-124709",
    "issue_type": "Service Request",
    "priority": "Low",
    "assignment_group": ""

})

print(result3)

{'issue_key': 'SEIT-124709', 'issue_type': 'Service Request', 'priority': 'Low', 'assignment_group': 'L1 Support Team'}


In [24]:
import pandas as pd

results = [
    result1,
    result2,
    result3
]

df = pd.DataFrame(results)

df

,issue_key,issue_type,priority,assignment_group
0,SEIT-124577,Incident,High,L2 Support Team
1,SEIT-124711,Incident,Low,L1 Support Team
2,SEIT-124709,Service Request,Low,L1 Support Team


#### Concept 4: Conditional looping

In [19]:
from IPython.display import display, HTML


conditional_looping_html = """
<div style="font-family:Segoe UI,Arial,sans-serif;line-height:1.6;">

    <h2 style="color:#0078d4;">
        Concept 4 - Conditional Looping
    </h2>

    <p>
        This workflow demonstrates how LangGraph can repeatedly search for a solution
        until an incident is resolved.
    </p>

    <h3>Business Scenario</h3>

    <p>
        When a Jira Incident is raised, the system analyzes the ticket.
        If a solution is not found, it searches the Knowledge Base (KB)
        and retries the analysis.
    </p>

    <pre style="background:#f4f4f4;padding:10px;">
Analyze Ticket
      ↓
Solution Found?
   /         \\
 Yes         No
  ↓           ↓
Resolve    Search KB
              ↓
         Analyze Again
              ↺
    </pre>

    <h3>Example</h3>

    <ul>
        <li>Attempt 1 → No Solution Found</li>
        <li>Attempt 2 → No Solution Found</li>
        <li>Attempt 3 → Solution Found</li>
        <li>Ticket Resolved</li>
    </ul>

    <div style="background:#dff6dd;padding:12px;border-left:4px solid #107c10;border-radius:4px;">
        <strong>
            Key Learning:
        </strong>
        LangGraph loops allow AI systems to retry actions until a desired
        outcome is achieved, making workflows intelligent and self-correcting.
    </div>

</div>
"""
display(HTML(conditional_looping_html))

In [15]:
from typing import TypedDict

class IncidentState(TypedDict):
    issue_key: str
    summary: str
    resolved: bool
    attempts: int
    solution: str
def analyze_ticket(state):

    print(f"Analyzing Ticket: {state['issue_key']}")

    return state


def kb_search(state):

    state["attempts"] += 1

    print(f"KB Search Attempt: {state['attempts']}")

    # Simulate finding solution after 3 attempts

    if state["attempts"] >= 3:
        state["resolved"] = True

    return state


def resolve_ticket(state):

    state["solution"] = "Known KB Resolution Applied"

    print("Ticket Resolved")

    return state
def routing_decision(state):

    if state["resolved"]:
        return "resolve_ticket"

    return "kb_search"
from langgraph.graph import StateGraph

builder = StateGraph(IncidentState)

builder.add_node(
    "analyze_ticket",
    analyze_ticket
)

builder.add_node(
    "kb_search",
    kb_search
)

builder.add_node(
    "resolve_ticket",
    resolve_ticket
)

builder.set_entry_point(
    "analyze_ticket"
)

builder.add_conditional_edges(
    "analyze_ticket",
    routing_decision,
    {
        "kb_search": "kb_search",
        "resolve_ticket": "resolve_ticket"
    }
)

builder.add_edge(
    "kb_search",
    "analyze_ticket"
)

builder.set_finish_point(
    "resolve_ticket"
)
graph = builder.compile()
from IPython.display import Image, Markdown, display

try:

    display(
        Image(
            graph.get_graph().draw_mermaid_png()
        )
    )

except Exception as e:

    print(f"Graph Visualization Failed: {e}")

    mermaid_code = graph.get_graph().draw_mermaid()

    display(
        Markdown(
            f"```mermaid\n{mermaid_code}\n```"
        )
    )


Graph Visualization Failed: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	analyze_ticket(analyze_ticket)
	kb_search(kb_search)
	resolve_ticket(resolve_ticket)
	__end__([<p>__end__</p>]):::last
	__start__ --> analyze_ticket;
	analyze_ticket -.-> kb_search;
	analyze_ticket -.-> resolve_ticket;
	kb_search --> analyze_ticket;
	resolve_ticket --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [16]:
result = graph.invoke({
    "issue_key": "SEIT-124577",
    "summary": "P1 DM not available",
    "resolved": False,
    "attempts": 0,
    "solution": ""
})

Analyzing Ticket: SEIT-124577
KB Search Attempt: 1
Analyzing Ticket: SEIT-124577
KB Search Attempt: 2
Analyzing Ticket: SEIT-124577
KB Search Attempt: 3
Analyzing Ticket: SEIT-124577
Ticket Resolved


In [17]:
print(result)
import pandas as pd

result_df = pd.DataFrame([result])

result_df

{'issue_key': 'SEIT-124577', 'summary': 'P1 DM not available', 'resolved': True, 'attempts': 3, 'solution': 'Known KB Resolution Applied'}


,issue_key,summary,resolved,attempts,solution
0,SEIT-124577,P1 DM not available,True,3,Known KB Resolution Applied


#### Concept 5: Conditional Skip

In [25]:
from IPython.display import display, HTML
conditional_skip_html = """
<div style="font-family:Segoe UI,Arial,sans-serif;line-height:1.6;">

    <h2 style="color:#0078d4;">
        Concept 5 - Conditional Skip
    </h2>

    <p>
        This example demonstrates how LangGraph can dynamically skip
        unnecessary workflow steps.
    </p>

    <h3>Business Scenario</h3>

    <p>
        When a Jira Incident is raised, the workflow first checks whether
        it is a Known Issue.
    </p>

    <ul>
        <li>If it is a Known Issue, skip Knowledge Base search and resolve immediately.</li>
        <li>If it is not a Known Issue, search the Knowledge Base before resolving.</li>
    </ul>

    <pre style="background:#f4f4f4;padding:10px;">
Incident Raised
      ↓
Known Issue?
   /        \\
 Yes         No
  ↓           ↓
Resolve    Search KB
              ↓
          Resolve
    </pre>

    <div style="background:#dff6dd;
                padding:12px;
                border-left:4px solid #107c10;
                border-radius:4px;">

        <strong>Key Learning:</strong><br>

        LangGraph can intelligently skip nodes based on state values,
        reducing unnecessary processing and improving workflow efficiency.

    </div>

</div>
"""

display(HTML(conditional_skip_html))

In [26]:
from typing import TypedDict

class TicketState(TypedDict):

    issue_key: str
    summary: str
    known_issue: bool
    resolution: str

def start(state):

    print(f"Processing Ticket: {state['issue_key']}")

    return state


def kb_search(state):

    print("Searching Knowledge Base...")

    state["resolution"] = (
        "Resolution identified from KB Article"
    )

    return state


def resolve_ticket(state):

    if state["resolution"] == "":

        state["resolution"] = (
            "Direct Resolution Applied"
        )

    print("Ticket Resolved")

    return state
def route_ticket(state):

    if state["known_issue"]:

        return "resolve_ticket"

    return "kb_search"
from langgraph.graph import StateGraph

builder = StateGraph(TicketState)

builder.add_node(
    "start",
    start
)

builder.add_node(
    "kb_search",
    kb_search
)

builder.add_node(
    "resolve_ticket",
    resolve_ticket
)

builder.set_entry_point(
    "start"
)

builder.add_conditional_edges(
    "start",
    route_ticket,
    {
        "resolve_ticket": "resolve_ticket",
        "kb_search": "kb_search"
    }
)

builder.add_edge(
    "kb_search",
    "resolve_ticket"
)

builder.set_finish_point(
    "resolve_ticket"
)
graph = builder.compile()
from IPython.display import Image, Markdown, display

try:

    display(
        Image(
            graph.get_graph().draw_mermaid_png()
        )
    )

except Exception as e:

    print(f"Graph Visualization Failed: {e}")

    mermaid_code = graph.get_graph().draw_mermaid()

    display(
        Markdown(
            f"```mermaid\n{mermaid_code}\n```"
        )
    )


Graph Visualization Failed: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start(start)
	kb_search(kb_search)
	resolve_ticket(resolve_ticket)
	__end__([<p>__end__</p>]):::last
	__start__ --> start;
	kb_search --> resolve_ticket;
	start -.-> kb_search;
	start -.-> resolve_ticket;
	resolve_ticket --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [27]:
result1 = graph.invoke({

    "issue_key": "SEIT-124577",
    "summary": "P1 DM not available",
    "known_issue": True,
    "resolution": ""

})

print(result1)

Processing Ticket: SEIT-124577
Ticket Resolved
{'issue_key': 'SEIT-124577', 'summary': 'P1 DM not available', 'known_issue': True, 'resolution': 'Direct Resolution Applied'}


In [28]:
result2 = graph.invoke({

    "issue_key": "SEIT-124711",
    "summary": "Kan inte ladda upp filer i DM värme eDOCS",
    "known_issue": False,
    "resolution": ""

})

print(result2)

Processing Ticket: SEIT-124711
Searching Knowledge Base...
Ticket Resolved
{'issue_key': 'SEIT-124711', 'summary': 'Kan inte ladda upp filer i DM värme eDOCS', 'known_issue': False, 'resolution': 'Resolution identified from KB Article'}


In [29]:
import pandas as pd

results = [
    result1,
    result2
]

df = pd.DataFrame(results)

df

,issue_key,summary,known_issue,resolution
0,SEIT-124577,P1 DM not available,True,Direct Resolution Applied
1,SEIT-124711,Kan inte ladda upp filer i DM värme eDOCS,False,Resolution identified from KB Article
